# Fine-Tuning with LoRA (Low-Rank Adaptation)

Full fine-tuning of large models is prohibitively expensive. LoRA offers a **parameter-efficient**
alternative. This notebook covers:
1. **Why parameter-efficient fine-tuning** (PEFT)
2. **LoRA mathematics** -- low-rank decomposition
3. **Implementation** from scratch
4. **Parameter savings** analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. The Fine-Tuning Problem

| Model | Parameters | GPU Memory (FP16) |
|-------|-----------|-------------------|
| GPT-2 | 1.5B | ~3 GB |
| LLaMA-7B | 7B | ~14 GB |
| LLaMA-70B | 70B | ~140 GB |

**Full fine-tuning** updates all parameters: $W' = W + \Delta W$ where $\Delta W \in \mathbb{R}^{d \times d}$.

**Key insight (Aghajanyan et al., 2020):** the weight updates $\Delta W$ during fine-tuning
have a **low intrinsic rank** -- we don't need all $d^2$ parameters.

In [ ]:
# Demonstrate low intrinsic dimensionality
# Simulate a weight update matrix and check its singular values
d = 256
# A "realistic" weight update: structured, not random
true_rank = 4
A_true = np.random.randn(d, true_rank) * 0.01
B_true = np.random.randn(true_rank, d) * 0.01
delta_W = A_true @ B_true + np.random.randn(d, d) * 0.0001  # low rank + small noise

# SVD to check rank
U, S, Vt = np.linalg.svd(delta_W)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(S[:50], 'o-', markersize=3)
axes[0].set_xlabel('Singular value index')
axes[0].set_ylabel('Singular value (log scale)')
axes[0].set_title('Singular Value Spectrum of ΔW')
axes[0].axvline(true_rank, ls='--', color='red', label=f'True rank = {true_rank}')
axes[0].legend()

# Cumulative energy
energy = np.cumsum(S**2) / np.sum(S**2)
axes[1].plot(energy[:50], 'o-', markersize=3)
axes[1].axhline(0.99, ls='--', color='gray', label='99% energy')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative energy')
axes[1].set_title('ΔW is effectively low-rank')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. LoRA: The Key Idea

Instead of learning $\Delta W \in \mathbb{R}^{d \times d}$, decompose it as:

$$\Delta W = B A, \quad B \in \mathbb{R}^{d \times r}, \quad A \in \mathbb{R}^{r \times d}$$

where $r \ll d$ (typically $r = 4, 8, 16$).

**Forward pass:** $h = (W + \Delta W) x = W x + B A x$

- $W$ is **frozen** (no gradients)
- Only $A$ and $B$ are trained
- Parameters: $2 \cdot d \cdot r$ instead of $d^2$

In [ ]:
class LoRALinear:
    """A linear layer with LoRA adaptation."""
    def __init__(self, d_in, d_out, rank=4, alpha=1.0):
        # Frozen pretrained weight
        self.W = np.random.randn(d_out, d_in) * 0.02
        
        # LoRA adapters
        self.A = np.random.randn(rank, d_in) * 0.01   # down-projection
        self.B = np.zeros((d_out, rank))                # up-projection (init to zero)
        
        self.rank = rank
        self.alpha = alpha  # scaling factor
        self.scaling = alpha / rank
    
    def forward(self, x):
        """x: shape (batch, d_in)"""
        # Original path (frozen)
        h = x @ self.W.T
        # LoRA path (trainable)
        lora_out = x @ self.A.T @ self.B.T * self.scaling
        return h + lora_out
    
    def merge(self):
        """Merge LoRA weights into the base weight for inference."""
        self.W += self.B @ self.A * self.scaling
        return self.W
    
    @property
    def n_trainable(self):
        return self.A.size + self.B.size
    
    @property
    def n_total(self):
        return self.W.size

# Example
d_in, d_out = 768, 768  # typical Transformer hidden size
rank = 8
layer = LoRALinear(d_in, d_out, rank=rank)

x = np.random.randn(4, d_in)  # batch of 4
output = layer.forward(x)

print(f'Input shape: {x.shape}')
print(f'Output shape: {output.shape}')
print(f'Original parameters: {layer.n_total:,}')
print(f'LoRA trainable parameters: {layer.n_trainable:,}')
print(f'Reduction: {layer.n_trainable / layer.n_total * 100:.2f}% of original')

## 3. Parameter Savings at Scale

In [ ]:
# Compare parameter counts for a Transformer model
d_model = 4096  # LLaMA-7B hidden size
n_layers = 32
n_attention_matrices = 4  # Q, K, V, O per layer

full_params = n_layers * n_attention_matrices * d_model * d_model

ranks = [1, 2, 4, 8, 16, 32, 64]
lora_params = [n_layers * n_attention_matrices * 2 * d_model * r for r in ranks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(ranks)), [p / 1e6 for p in lora_params], color='steelblue')
ax.axhline(full_params / 1e6, color='red', ls='--', lw=2, label=f'Full FT: {full_params/1e6:.0f}M')
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('Trainable Parameters (millions)')
ax.set_title('LoRA Parameter Count vs Full Fine-Tuning (LLaMA-7B scale)')
ax.legend()

for i, (r, p) in enumerate(zip(ranks, lora_params)):
    pct = p / full_params * 100
    ax.text(i, p / 1e6 + 5, f'{pct:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Pseudo-Code for LoRA Training with PyTorch

```python
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

# Configure LoRA
lora_config = LoraConfig(
    r=8,                    # rank
    lora_alpha=16,          # scaling
    target_modules=["q_proj", "v_proj"],  # which layers to adapt
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# Wrap model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# -> trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622%

# Train with standard loop or Trainer
# ... (same as full fine-tuning, but much faster)

# Save only LoRA weights (~17 MB vs ~14 GB)
model.save_pretrained("my_lora_adapter")
```

In [ ]:
# Demonstrate LoRA merging
layer = LoRALinear(768, 768, rank=8)
# Simulate training: update A and B
layer.A = np.random.randn(8, 768) * 0.01
layer.B = np.random.randn(768, 8) * 0.01

# Before merge
x_test = np.random.randn(2, 768)
out_before = layer.forward(x_test)

# After merge: single matrix, no extra computation at inference
W_merged = layer.merge()
out_after = x_test @ W_merged.T

print(f'Max difference after merging: {np.max(np.abs(out_before - out_after)):.2e}')
print('LoRA weights can be merged for zero-overhead inference.')
print('Multiple LoRA adapters can be swapped at serving time for different tasks.')

## Key Takeaways

- **LoRA** freezes the base model and trains low-rank adapter matrices $A$ and $B$.
- Typically trains **< 1%** of the original parameters with minimal quality loss.
- Adapters can be **merged** for zero-overhead inference or **swapped** for multi-task serving.
- Use `peft` library with HuggingFace for easy LoRA integration.
- Related methods: QLoRA (quantised base), AdaLoRA (adaptive rank), prefix tuning.

**Next:** Retrieval-Augmented Generation (RAG) -- combining retrieval with generation.